# AREx ARC-AGI-3 submission

In [ ]:
%pip install --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv
%pip install --no-index --find-links /kaggle/input/arex-kaggle-wheelhouse arc-agi-3 transformers accelerate safetensors


In [ ]:
%%writefile /tmp/my_agent.py
"""Official ARC-AGI-3 Agent wrapper around the AREx Kaggle bridge."""

from typing import Any

from agents.agent import Agent

from arc_agi_3.deployment import KaggleAgentMixin, KaggleSettings, build_kaggle_bridge


class MyAgent(KaggleAgentMixin, Agent):
    MAX_ACTIONS = 40

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        settings = KaggleSettings.from_env()
        self.MAX_ACTIONS = settings.max_actions
        self.arex_bridge = build_kaggle_bridge(self.game_id, settings=settings)


In [ ]:
import os
import subprocess
os.environ['AREX_MODEL_PATH'] = '/kaggle/input/qwen-3/transformers/8b/1'
os.environ['AREX_MAX_ACTIONS'] = '40'
os.environ['AREX_MAX_MODEL_CALLS'] = '4'
os.environ['AREX_MAX_NEW_TOKENS'] = '2048'
os.environ['AREX_MAX_REPAIRS'] = '1'

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    gateway = 'http://gateway:8001/api/games'
    subprocess.run(['curl', '--fail', '--retry', '120', '--retry-all-errors', '--retry-delay', '5', gateway], check=True)
    source = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/' + 'ARC-AGI-3-Agents'
    root = __import__('pathlib').Path('/kaggle/working/' + 'ARC-AGI-3-Agents')
    subprocess.run(['cp', '-r', source, str(root)], check=True)
    target = root / 'agents/templates/my_agent.py'
    subprocess.run(['cp', '/tmp/my_agent.py', str(target)], check=True)
    agents = "from .agent import Agent, Playback\nfrom .swarm import Swarm\nfrom .templates.my_agent import MyAgent\nAVAILABLE_AGENTS = {'myagent': MyAgent}\n"
    (root / 'agents/__init__.py').write_text(agents)
    env = 'SCHEME=http\nHOST=gateway\nPORT=8001\n' + 'ARC_API_KEY=test-key-123\nARC_BASE_URL=http://gateway:8001/\n' + 'OPERATION_MODE=online\nRECORDINGS_DIR=/kaggle/working/server_recording\n'
    (root / '.env').write_text(env)
    subprocess.run(['python', 'main.py', '--agent', 'myagent'], cwd=root, check=True)

In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    columns = ['row_id', 'game_id', 'end_of_game', 'score']
    submission = pd.DataFrame([['1_0', '1', True, 1]], columns=columns)
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
